## Exploratory Data Analysis of the dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
def dataset_overview(df, target=None, id_col="auto", cat_threshold=15,
                     save_html=None, name=None, max_sample=50_000):
    import warnings, io, base64, os
    warnings.filterwarnings("ignore")
    from IPython.display import display, HTML
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from scipy.stats import gaussian_kde

    COLORS = px.colors.qualitative.Plotly

    # ── Report name ───────────────────────────────────────────────────────────
    import os as _os
    if name is not None:
        _stem = _os.path.splitext(_os.path.basename(str(name)))[0]
        report_name = _stem.replace("_", " ").replace("-", " ").title()
    else:
        report_name = "Dataset Overview"

    html_parts = []

    # ── CSS ───────────────────────────────────────────────────────────────────
    CSS = """
    <style>
    .eda-section {
        border-left: 6px solid #2563EB !important;
        background: linear-gradient(90deg, #EFF6FF 0%, #F8FAFF 100%) !important;
        padding: 12px 20px; margin: 32px 0 14px;
        border-radius: 0 10px 10px 0;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .eda-section .sec-num {
        font-size: 11px; color: #6B7280 !important; font-weight: 700;
        letter-spacing: 1.5px; text-transform: uppercase;
    }
    .eda-section .sec-title {
        font-size: 20px; font-weight: 800; color: #1E3A5F !important; margin-top: 2px;
    }
    .eda-kv { display: flex; gap: 14px; flex-wrap: wrap; margin: 12px 0 20px; }
    .eda-kv-card {
        background: #FFFFFF !important; border: 1px solid #E5E7EB !important;
        border-radius: 10px; padding: 14px 22px; min-width: 150px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.06);
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .eda-kv-card .kv-label {
        font-size: 11px; color: #6B7280 !important; font-weight: 700;
        text-transform: uppercase; letter-spacing: 0.8px;
    }
    .eda-kv-card .kv-value {
        font-size: 24px; font-weight: 800; color: #2563EB !important; margin-top: 4px;
    }
    .eda-kv-card .kv-sub { font-size: 12px; color: #9CA3AF !important; margin-top: 2px; }
    table.eda-table {
        border-collapse: collapse; width: 100%; font-size: 13px;
        margin: 8px 0 20px; border-radius: 8px; overflow: hidden;
        box-shadow: 0 1px 4px rgba(0,0,0,0.08);
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    table.eda-table th {
        background: #2563EB !important; color: #FFFFFF !important;
        padding: 11px 16px; text-align: center !important; font-size: 13px; font-weight: 600;
    }
    table.eda-table td {
        padding: 9px 16px; border-bottom: 1px solid #F3F4F6 !important;
        color: #111827 !important; background: #FFFFFF !important; text-align: center !important;
    }
    table.eda-table tr:last-child td { border-bottom: none !important; }
    table.eda-table tr:nth-child(even) td {
        background: #F0F7FF !important; color: #111827 !important;
    }
    table.eda-table tr:hover td {
        background: #DBEAFE !important; color: #111827 !important;
    }
    .eda-sub {
        font-size: 15px; font-weight: 700; color: #1E3A5F !important;
        background: #EFF6FF !important;
        border-left: 4px solid #2563EB; border-bottom: 2px solid #BFDBFE;
        padding: 8px 14px; margin: 20px 0 8px; border-radius: 0 6px 6px 0;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .badge {
        display: inline-block; padding: 3px 10px; border-radius: 12px;
        font-size: 12px; font-weight: 700;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .badge-blue   { background: #DBEAFE !important; color: #1D4ED8 !important; }
    .badge-green  { background: #D1FAE5 !important; color: #065F46 !important; }
    .badge-orange { background: #FEF3C7 !important; color: #92400E !important; }
    .badge-red    { background: #FEE2E2 !important; color: #991B1B !important; }
    .badge-gray   { background: #F3F4F6 !important; color: #4B5563 !important; }
    .eda-alert {
        padding: 11px 16px; border-radius: 8px; font-size: 13px;
        font-weight: 600; margin: 8px 0;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }
    .alert-green {
        background: #D1FAE5 !important; color: #065F46 !important;
        border-left: 4px solid #10B981;
    }
    .alert-red {
        background: #FEE2E2 !important; color: #991B1B !important;
        border-left: 4px solid #EF4444;
    }
    .alert-blue {
        background: #DBEAFE !important; color: #1D4ED8 !important;
        border-left: 4px solid #3B82F6;
    }
    .alert-yellow {
        background: #FEF9C3 !important; color: #854D0E !important;
        border-left: 4px solid #EAB308;
    }
    </style>
    """
    display(HTML(CSS))
    html_parts.append(CSS)

    # ── Helpers ───────────────────────────────────────────────────────────────
    def _emit(html):
        display(HTML(html))
        html_parts.append(html)

    def sec(num, title):
        _emit(
            f'<div class="eda-section">'
            f'<div class="sec-num">Section {num}</div>'
            f'<div class="sec-title">{title}</div>'
            f'</div>'
        )

    def kv(data):
        cards = ""
        for label, val in data.items():
            sub_lbl = ""
            if isinstance(val, tuple):
                val, sub_lbl = val
                sub_lbl = f'<div class="kv-sub">{sub_lbl}</div>'
            cards += (
                f'<div class="eda-kv-card">'
                f'<div class="kv-label">{label}</div>'
                f'<div class="kv-value">{val}</div>{sub_lbl}'
                f'</div>'
            )
        _emit(f'<div class="eda-kv">{cards}</div>')

    # OPT: vectorised HTML table — avoids iterrows() O(n) Python loop
    def tbl(data):
        if isinstance(data, pd.Series):
            data = data.to_frame()
        df_r     = data if isinstance(data.index, pd.RangeIndex) else data.reset_index()
        th       = "".join(f"<th>{c}</th>" for c in df_r.columns)
        str_vals = df_r.astype(str).values          # single vectorised cast
        rows     = "".join(
            "<tr>" + "".join(f"<td>{cell}</td>" for cell in row) + "</tr>"
            for row in str_vals
        )
        _emit(f'<table class="eda-table"><thead><tr>{th}</tr></thead><tbody>{rows}</tbody></table>')

    def sub(title):
        _emit(f'<div class="eda-sub">{title}</div>')

    def alert(msg, kind="green"):
        _emit(f'<div class="eda-alert alert-{kind}">{msg}</div>')

    def badge(text, color="blue"):
        return f'<span class="badge badge-{color}">{text}</span>'

    _FONT = dict(family="Arial, sans-serif", color="#1F2937")

    def _style(fig, title="", height=500):
        fig.update_layout(
            title=dict(text=title,
                       font=dict(size=18, color="#1E3A5F", family="Arial, sans-serif"),
                       x=0.5, xanchor="center"),
            template="plotly_white",
            font=_FONT,
            paper_bgcolor="white",
            plot_bgcolor="white",
            height=height,
            margin=dict(t=80, b=60, l=70, r=40),
        )
        fig.update_xaxes(showgrid=False, zeroline=False,
                         linecolor="#E5E7EB", tickfont=dict(size=12))
        fig.update_yaxes(showgrid=False, zeroline=False,
                         linecolor="#E5E7EB", tickfont=dict(size=12))
        return fig

    def show_fig(fig):
        if save_html is not None:
            try:
                img_bytes = fig.to_image(format="png", scale=2)
                img_b64   = base64.b64encode(img_bytes).decode()
                html_parts.append(
                    f'<div style="text-align:center;margin:24px 0;">'
                    f'<img src="data:image/png;base64,{img_b64}" '
                    f'style="max-width:100%;border-radius:8px;'
                    f'box-shadow:0 2px 12px rgba(0,0,0,0.10);">'
                    f'</div>'
                )
            except Exception as e:
                html_parts.append(
                    f'<p style="color:#991B1B;font-family:sans-serif;">'
                    f'Chart export failed: {e}</p>'
                )
        fig.show()

    # ── Column Classification ─────────────────────────────────────────────────
    if id_col == "auto":
        id_cols = [
            c for c in df.columns
            if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique() == len(df)
        ]
    elif id_col is not None:
        id_cols = [id_col] if isinstance(id_col, str) else list(id_col)
    else:
        id_cols = []

    # OPT: drop() returns a new lightweight df — avoids a full df.copy()
    df = df.drop(columns=id_cols, errors="ignore")

    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    promoted = [
        c for c in num_cols
        if c != target
        and pd.api.types.is_integer_dtype(df[c])
        and df[c].nunique() <= cat_threshold
    ]
    num_cols = [c for c in num_cols if c not in promoted]
    cat_cols = promoted + cat_cols

    # OPT: precompute value_counts once per categorical column — reused in §5 and §11
    cat_vc = {col: df[col].value_counts(dropna=False) for col in cat_cols}

    # OPT: build plot sample once — all visualisations reuse df_plot
    n        = len(df)
    is_large = n > max_sample
    if is_large:
        df_plot = df.sample(n=max_sample, random_state=42)
        sample_note = (
            f"&#9432; Large dataset detected ({n:,} rows). "
            f"Statistics computed on all {n:,} rows. "
            f"Visualisations use a random sample of {max_sample:,} rows."
        )
    else:
        df_plot      = df
        sample_note  = ""

    # boxpoints strategy: show all points only when sample is small enough
    box_pts = "all" if len(df_plot) <= 5_000 else "outliers"

    # ── 0. Column Classification ──────────────────────────────────────────────
    sec(0, "Column Classification")
    if is_large:
        alert(sample_note, "yellow")
    dropped_html  = (", ".join(badge(c, "red")    for c in id_cols))  if id_cols  else badge("none", "gray")
    promoted_html = (", ".join(badge(c, "orange") for c in promoted)) if promoted else badge("none", "gray")
    num_html = " ".join(badge(c, "blue")  for c in num_cols)
    cat_html = " ".join(badge(c, "green") for c in cat_cols)
    _emit(
        '<table class="eda-table"><thead><tr>'
        '<th>Decision</th><th>Columns</th><th>Reason</th>'
        '</tr></thead><tbody>'
        f'<tr><td>Primary key dropped</td><td>{dropped_html}</td><td>All values unique (nunique == nrows)</td></tr>'
        f'<tr><td>Numeric &#8594; Categorical</td><td>{promoted_html}</td><td>Integer dtype with &le; {cat_threshold} unique values</td></tr>'
        f'<tr><td>Numerical columns</td><td>{num_html}</td><td>Continuous numeric</td></tr>'
        f'<tr><td>Categorical columns</td><td>{cat_html}</td><td>Object / category / low-cardinality int</td></tr>'
        '</tbody></table>'
    )

    # ── 1. Basic Information ──────────────────────────────────────────────────
    sec(1, "Basic Information")
    mem_kb = df.memory_usage(deep=True).sum() / 1024
    kv({
        "Rows":        (f"{n:,}",              "observations"),
        "Columns":     (str(df.shape[1]),       "excl. primary key"),
        "Numerical":   (str(len(num_cols)),     "features"),
        "Categorical": (str(len(cat_cols)),     "features"),
        "Memory":      (f"{mem_kb:.1f} KB",     "deep usage"),
    })
    sub("Column Dtypes — Stored vs Treated As")
    dtype_df = df.dtypes.rename("Stored Dtype").to_frame()
    dtype_df.index.name = "Column"
    dtype_df["Treated As"] = [
        "Categorical" if c in cat_cols else "Numerical" for c in dtype_df.index
    ]
    tbl(dtype_df)

    # ── 2. Missing Values ─────────────────────────────────────────────────────
    sec(2, "Missing Values")
    # OPT: single isnull() scan — derive % from count instead of a second mean() pass
    null_counts = df.isnull().sum()
    missing = pd.DataFrame({
        "Missing Count": null_counts,
        "Missing %":     (null_counts / n * 100).round(2),
    }).query("`Missing Count` > 0")
    missing.index.name = "Column"
    if missing.empty:
        alert("&#10003; No missing values found across all columns.", "green")
    else:
        tbl(missing)

    # ── 3. Duplicates ─────────────────────────────────────────────────────────
    sec(3, "Duplicate Rows")
    n_dup = df.duplicated().sum()
    if n_dup == 0:
        alert("&#10003; No duplicate rows found.", "green")
    else:
        alert(f"&#9888; {n_dup:,} duplicate rows detected ({n_dup/n*100:.2f}% of dataset).", "red")

    # ── 4. Numerical Summary ──────────────────────────────────────────────────
    sec(4, "Numerical Summary")
    if num_cols:
        # OPT: assign num_df once — avoids re-selecting the column subset 3×
        num_df = df[num_cols]
        desc   = num_df.describe().T.round(2)
        desc.index.name = "Feature"
        desc["skewness"] = num_df.skew().round(3)
        desc["kurtosis"] = num_df.kurt().round(3)
        tbl(desc)
    else:
        alert("No numerical columns after classification.", "blue")

    # ── 5. Categorical Summary ────────────────────────────────────────────────
    sec(5, "Categorical Summary")
    for col in cat_cols:
        tag = f'&nbsp;{badge("reclassified", "orange")}' if col in promoted else ""
        sub(f"{col} &nbsp;&#183;&nbsp; {df[col].nunique()} unique values{tag}")
        # OPT: reuse precomputed cat_vc — no duplicate value_counts() call
        vc  = cat_vc[col]
        pct = (vc / n * 100).round(2)
        tbl(pd.DataFrame({
            "Category":     vc.index.astype(str),
            "Count":        vc.values,
            "Percentage %": pct.values,
        }))

    # ── 6. Target Distribution ────────────────────────────────────────────────
    if target and target in df.columns:
        sec(6, f"Target Distribution — {target}")
        # OPT: reuse from cat_vc if target is categorical, else compute once
        vc  = cat_vc.get(target, df[target].value_counts(dropna=False))
        pct = (vc / n * 100).round(2)
        tbl(pd.DataFrame({
            "Class":        vc.index.astype(str),
            "Count":        vc.values,
            "Percentage %": pct.values,
        }))
        # cache target_vals for reuse in §12a and §12b
        target_vals = df[target].dropna().unique().tolist()

        labels = [str(v) for v in vc.index]
        fig = go.Figure(go.Bar(
            x=labels,
            y=vc.values,
            marker_color=COLORS[:len(labels)],
            text=[f"{int(v):,}<br>({p:.1f}%)" for v, p in zip(vc.values, pct.values)],
            textposition="outside",
            textfont=dict(size=13, color="#1F2937"),
        ))
        _style(fig, title=f"Target Distribution — {target}", height=480)
        fig.update_layout(showlegend=False)
        fig.update_yaxes(tickformat=".2s")
        show_fig(fig)
    else:
        target_vals = []

    # ── 7. Outlier Detection ──────────────────────────────────────────────────
    sec(7, "Outlier Detection — IQR Method")
    if num_cols:
        # OPT: compute all quantiles in one vectorised call
        q1_all = num_df.quantile(0.25)
        q3_all = num_df.quantile(0.75)
        iqr_all = q3_all - q1_all
        lo_all  = q1_all - 1.5 * iqr_all
        hi_all  = q3_all + 1.5 * iqr_all
        rows = []
        for col in num_cols:
            n_out = ((df[col] < lo_all[col]) | (df[col] > hi_all[col])).sum()
            rows.append({
                "Column":        col,
                "Q1":            round(q1_all[col], 2),
                "Q3":            round(q3_all[col], 2),
                "IQR":           round(iqr_all[col], 2),
                "Lower Fence":   round(lo_all[col], 2),
                "Upper Fence":   round(hi_all[col], 2),
                "Outlier Count": int(n_out),
                "Outlier %":     round(n_out / n * 100, 2),
            })
        tbl(pd.DataFrame(rows).set_index("Column"))
    else:
        alert("No numerical columns after classification.", "blue")

    # ── 8. Correlation Matrix ─────────────────────────────────────────────────
    if len(num_cols) > 1:
        sec(8, "Correlation Matrix")
        corr = num_df.corr().round(2)
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
        z    = corr.where(~mask).values
        fig  = go.Figure(go.Heatmap(
            z=z,
            x=corr.columns.tolist(),
            y=corr.index.tolist(),
            colorscale="RdBu_r",
            zmid=0, zmin=-1, zmax=1,
            text=corr.where(~mask).round(2).values,
            texttemplate="%{text}",
            textfont=dict(size=11, color="#1F2937"),
            hoverongaps=False,
            colorbar=dict(thickness=16, len=0.8),
        ))
        side = max(500, len(num_cols) * 70)
        _style(fig, title="Correlation Matrix — Numerical Features", height=side)
        fig.update_layout(margin=dict(t=80, b=80, l=100, r=40))
        show_fig(fig)

    # ── 9. Scatter Plot Matrix ────────────────────────────────────────────────
    if len(num_cols) > 1:
        sec(9, "Scatter Plot Matrix")
        # OPT: cap columns at 8 to avoid browser overload; use df_plot (sampled)
        scatter_cols = num_cols[:8]
        hue_col      = target if (target and target in df_plot.columns) else None
        plot_cols    = scatter_cols + ([hue_col] if hue_col else [])
        if is_large:
            alert(f"&#9432; Scatter matrix uses {len(df_plot):,}-row sample.", "yellow")
        fig = px.scatter_matrix(
            df_plot[plot_cols],
            dimensions=scatter_cols,
            color=hue_col,
            color_discrete_sequence=COLORS,
            opacity=0.35,
            labels={c: c.replace("_", " ") for c in scatter_cols},
        )
        fig.update_traces(marker=dict(size=4, line=dict(width=0)), diagonal_visible=True)
        grid_size = max(700, len(scatter_cols) * 130)
        _style(fig, title="Scatter Plot Matrix — Numerical Features", height=grid_size)
        fig.update_layout(width=grid_size, margin=dict(t=80, b=60, l=60, r=40))
        show_fig(fig)

    # ── 10. Numerical Distributions ───────────────────────────────────────────
    if num_cols:
        sec(10, "Numerical Distributions")
        if is_large:
            alert(f"&#9432; Distributions use {len(df_plot):,}-row sample.", "yellow")
        ncols_g = 3
        nrows_g = -(-len(num_cols) // ncols_g)
        # OPT: compute skewness from full num_df (already assigned); no extra scan
        skew_vals = num_df.skew()
        titles    = [f"{c}<br><sup>skew = {skew_vals[c]:.2f}</sup>" for c in num_cols]
        titles   += [""] * (nrows_g * ncols_g - len(num_cols))
        fig = make_subplots(rows=nrows_g, cols=ncols_g, subplot_titles=titles)

        for i, col in enumerate(num_cols):
            r, c_idx = i // ncols_g + 1, i % ncols_g + 1
            color    = COLORS[i % len(COLORS)]
            # OPT: dropna on sampled df_plot — gaussian_kde on ≤ max_sample points
            data     = df_plot[col].dropna().values
            fig.add_trace(
                go.Histogram(
                    x=data, histnorm="probability density",
                    marker_color=color, opacity=0.70,
                    marker_line=dict(color="white", width=0.5),
                    showlegend=False, name=col,
                ),
                row=r, col=c_idx,
            )
            kde     = gaussian_kde(data)
            x_range = np.linspace(data.min(), data.max(), 300)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=kde(x_range), mode="lines",
                    line=dict(color="#1E3A5F", width=2.5),
                    showlegend=False,
                ),
                row=r, col=c_idx,
            )
            axis_key = f"xaxis{'' if i == 0 else i + 1}"
            fig.update_layout({axis_key: dict(tickformat=".2s", showgrid=False, zeroline=False)})

        _style(fig, height=420 * nrows_g)
        fig.update_layout(
            title=dict(text="Numerical Feature Distributions",
                       font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
        )
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=False, zeroline=False, title_text="Density")
        show_fig(fig)

    # ── 11. Categorical Distributions ────────────────────────────────────────
    if cat_cols:
        sec(11, "Categorical Distributions")
        ncols_g = min(3, len(cat_cols))
        nrows_g = -(-len(cat_cols) // ncols_g)
        titles  = [f"{c}{' [reclassified]' if c in promoted else ''}" for c in cat_cols]
        titles += [""] * (nrows_g * ncols_g - len(cat_cols))
        fig     = make_subplots(rows=nrows_g, cols=ncols_g, subplot_titles=titles)

        for i, col in enumerate(cat_cols):
            r, c_idx = i // ncols_g + 1, i % ncols_g + 1
            # OPT: reuse precomputed cat_vc — no duplicate value_counts()
            vc       = cat_vc[col]
            x_vals   = [str(v) for v in vc.index]
            y_vals   = vc.values
            bar_clrs = [COLORS[j % len(COLORS)] for j in range(len(x_vals))]
            fig.add_trace(
                go.Bar(
                    x=x_vals, y=y_vals,
                    marker_color=bar_clrs,
                    marker_line=dict(color="white", width=0.6),
                    text=[f"{int(v):,}<br>({v/n*100:.1f}%)" for v in y_vals],
                    textposition="outside",
                    textfont=dict(size=11, color="#1F2937"),
                    showlegend=False,
                ),
                row=r, col=c_idx,
            )

        _style(fig, height=420 * nrows_g)
        fig.update_layout(
            title=dict(text="Categorical Feature Distributions",
                       font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
        )
        fig.update_xaxes(showgrid=False, zeroline=False, tickangle=30)
        fig.update_yaxes(showgrid=False, zeroline=False, title_text="Count")
        show_fig(fig)

    # ── 12a. Numerical Features vs Target ─────────────────────────────────────
    if target and target in df.columns and num_cols and target_vals:
        feat_cols = [c for c in num_cols if c != target]
        if feat_cols:
            sec("12a", f"Numerical Features vs Target — {target}")
            if is_large:
                alert(f"&#9432; Boxplots use {len(df_plot):,}-row sample.", "yellow")
            ncols_g = 3
            nrows_g = -(-len(feat_cols) // ncols_g)
            titles  = feat_cols + [""] * (nrows_g * ncols_g - len(feat_cols))
            fig     = make_subplots(rows=nrows_g, cols=ncols_g, subplot_titles=titles)

            for i, col in enumerate(feat_cols):
                r, c_idx = i // ncols_g + 1, i % ncols_g + 1
                for j, tval in enumerate(target_vals):
                    # OPT: filter on df_plot (sampled) — avoids sending all rows to browser
                    subset = df_plot[df_plot[target] == tval][col].dropna()
                    fig.add_trace(
                        go.Box(
                            y=subset,
                            name=str(tval),
                            marker_color=COLORS[j % len(COLORS)],
                            # OPT: "outliers" on large samples, "all" on small ones
                            boxpoints=box_pts,
                            jitter=0.35,
                            pointpos=0,
                            marker=dict(size=3, opacity=0.25),
                            line=dict(width=1.5),
                            showlegend=(i == 0),
                            legendgroup=str(tval),
                        ),
                        row=r, col=c_idx,
                    )

            _style(fig, height=480 * nrows_g)
            fig.update_layout(
                title=dict(text=f"Numerical Features by Target — {target}",
                           font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
                boxmode="group",
                legend=dict(title=dict(text=target, font=dict(size=13)),
                            font=dict(size=12)),
            )
            fig.update_xaxes(showgrid=False, zeroline=False)
            fig.update_yaxes(showgrid=False, zeroline=False, tickformat=".2s")
            show_fig(fig)

    # ── 12b. Reclassified Features vs Target ──────────────────────────────────
    if target and target in df.columns and target_vals:
        reclassified_feats = [c for c in promoted if c != target]
        if reclassified_feats:
            sec("12b", f"Reclassified Categorical Features vs Target — {target}")
            ncols_g = min(3, len(reclassified_feats))
            nrows_g = -(-len(reclassified_feats) // ncols_g)
            titles  = [f"{c} [reclassified]" for c in reclassified_feats]
            titles += [""] * (nrows_g * ncols_g - len(reclassified_feats))
            fig     = make_subplots(rows=nrows_g, cols=ncols_g, subplot_titles=titles)

            for i, col in enumerate(reclassified_feats):
                r, c_idx = i // ncols_g + 1, i % ncols_g + 1
                order    = sorted(df[col].dropna().unique())
                for j, tval in enumerate(target_vals):
                    # OPT: use precomputed cat_vc grouped by target — full df for counts
                    subset = df[df[target] == tval]
                    counts = [int((subset[col] == v).sum()) for v in order]
                    fig.add_trace(
                        go.Bar(
                            x=[str(v) for v in order],
                            y=counts,
                            name=str(tval),
                            marker_color=COLORS[j % len(COLORS)],
                            marker_line=dict(color="white", width=0.5),
                            showlegend=(i == 0),
                            legendgroup=str(tval),
                        ),
                        row=r, col=c_idx,
                    )

            _style(fig, height=420 * nrows_g)
            fig.update_layout(
                title=dict(text=f"Reclassified Features by Target — {target}",
                           font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
                barmode="group",
                legend=dict(title=dict(text=target, font=dict(size=13)),
                            font=dict(size=12)),
            )
            fig.update_xaxes(showgrid=False, zeroline=False)
            fig.update_yaxes(showgrid=False, zeroline=False, title_text="Count")
            show_fig(fig)

    # ── End banner ────────────────────────────────────────────────────────────
    _emit(
        '<div style="border-left:6px solid #10B981 !important;'
        ' background:#D1FAE5 !important; padding:12px 20px;'
        ' margin:32px 0 10px; border-radius:0 10px 10px 0;'
        ' font-family:sans-serif; font-size:15px; font-weight:700;'
        ' color:#065F46 !important;">'
        '&#10003; Dataset Overview Complete'
        '</div>'
    )

    # ── Save HTML file ────────────────────────────────────────────────────────
    if save_html is not None:
        out_path = os.path.abspath(save_html)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        full_html = (
            '<!DOCTYPE html>\n<html lang="en">\n<head>\n'
            '<meta charset="UTF-8">\n'
            '<meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
            f'<title>{report_name}</title>\n'
            '<style>\n'
            '  body { margin:0; padding:40px 20px; background:#F9FAFB;'
            ' font-family:-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; }\n'
            '  .report-wrap { max-width:1200px; margin:0 auto; background:white;'
            ' border-radius:16px; box-shadow:0 4px 24px rgba(0,0,0,0.08);'
            ' padding:48px 48px 60px; }\n'
            '  .report-title { font-size:32px; font-weight:900; color:#1E3A5F;'
            ' border-bottom:3px solid #2563EB; padding-bottom:16px; margin-bottom:32px; }\n'
            '</style>\n'
            '</head>\n<body>\n<div class="report-wrap">\n'
            f'<div class="report-title">&#128202; {report_name}</div>\n'
            + "\n".join(html_parts)
            + '\n</div>\n</body>\n</html>'
        )
        with open(out_path, "w", encoding="utf-8") as fh:
            fh.write(full_html)
        display(HTML(
            f'<div class="eda-alert alert-blue">'
            f'&#128190; Report saved to <strong>{out_path}</strong>'
            f'</div>'
        ))

In [3]:
# load dataset

data_path = Path("../dataset/loan_approval_dataset.csv")

print("Dataset exists : ", data_path.exists())

Dataset exists :  True


In [4]:
df = pd.read_csv(data_path, skipinitialspace=True)
dataset_overview(df, target="loan_status", save_html="../outputs/eda_report.html", name=data_path)

Decision,Columns,Reason
Primary key dropped,loan_id,All values unique (nunique == nrows)
Numeric → Categorical,"no_of_dependents, loan_term",Integer dtype with ≤ 15 unique values
Numerical columns,income_annum loan_amount cibil_score residential_assets_value commercial_assets_value luxury_assets_value bank_asset_value,Continuous numeric
Categorical columns,no_of_dependents loan_term education self_employed loan_status,Object / category / low-cardinality int


Column,Stored Dtype,Treated As
no_of_dependents,int64,Categorical
education,object,Categorical
self_employed,object,Categorical
income_annum,int64,Numerical
loan_amount,int64,Numerical
loan_term,int64,Categorical
cibil_score,int64,Numerical
residential_assets_value,int64,Numerical
commercial_assets_value,int64,Numerical
luxury_assets_value,int64,Numerical


Feature,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis
income_annum,4269.0,5059123.92,2806839.83,200000.0,2700000.0,5100000.0,7500000.0,9900000.0,-0.013,-1.183
loan_amount,4269.0,15133450.46,9043362.98,300000.0,7700000.0,14500000.0,21500000.0,39500000.0,0.309,-0.744
cibil_score,4269.0,599.94,172.43,300.0,453.0,600.0,748.0,900.0,-0.009,-1.186
residential_assets_value,4269.0,7472616.54,6503636.59,-100000.0,2200000.0,5600000.0,11300000.0,29100000.0,0.978,0.185
commercial_assets_value,4269.0,4973155.31,4388966.09,0.0,1300000.0,3700000.0,7600000.0,19400000.0,0.958,0.101
luxury_assets_value,4269.0,15126305.93,9103753.67,300000.0,7500000.0,14600000.0,21700000.0,39200000.0,0.322,-0.738
bank_asset_value,4269.0,4976692.43,3250185.31,0.0,2300000.0,4600000.0,7100000.0,14700000.0,0.561,-0.397


Category,Count,Percentage %
4,752,17.62
3,727,17.03
0,712,16.68
2,708,16.58
1,697,16.33
5,673,15.76


Category,Count,Percentage %
6,490,11.48
12,456,10.68
4,447,10.47
10,436,10.21
18,422,9.89
16,412,9.65
20,411,9.63
14,405,9.49
2,404,9.46
8,386,9.04


Category,Count,Percentage %
Graduate,2144,50.22
Not Graduate,2125,49.78


Category,Count,Percentage %
Yes,2150,50.36
No,2119,49.64


Category,Count,Percentage %
Approved,2656,62.22
Rejected,1613,37.78


Class,Count,Percentage %
Approved,2656,62.22
Rejected,1613,37.78


Column,Q1,Q3,IQR,Lower Fence,Upper Fence,Outlier Count,Outlier %
income_annum,2700000.0,7500000.0,4800000.0,-4500000.0,14700000.0,0,0.0
loan_amount,7700000.0,21500000.0,13800000.0,-13000000.0,42200000.0,0,0.0
cibil_score,453.0,748.0,295.0,10.5,1190.5,0,0.0
residential_assets_value,2200000.0,11300000.0,9100000.0,-11450000.0,24950000.0,52,1.22
commercial_assets_value,1300000.0,7600000.0,6300000.0,-8150000.0,17050000.0,37,0.87
luxury_assets_value,7500000.0,21700000.0,14200000.0,-13800000.0,43000000.0,0,0.0
bank_asset_value,2300000.0,7100000.0,4800000.0,-4900000.0,14300000.0,5,0.12


## Key EDA Findings

### Dataset Overview
- **4,269 rows × 13 columns** — no missing values, no duplicates
- **Target:** `loan_status` — Approved 62.2% | Rejected 37.8% (mild class imbalance)
- `loan_id` auto-dropped (primary key), `no_of_dependents` and `loan_term` reclassified from int → categorical

---

### Finding 1 — CIBIL Score is the Dominant Predictor
CIBIL score has a **0.771 correlation** with the target. Every other numerical feature sits between −0.015 and +0.016.

| Loan Status | Mean CIBIL | Std |
|---|---|---|
| Approved | 703.5 | 125.2 |
| Rejected | 429.5 | 78.4 |

A threshold around 550–600 already separates most applicants. This will likely be the most important feature in any model.

---

### Finding 2 — Financial Features Have Near-Zero Individual Predictive Power
Income, loan amount, and all asset values show almost identical means across Approved and Rejected applicants:

| Feature | Approved Mean | Rejected Mean |
|---|---|---|
| `income_annum` | 5,025,904 | 5,113,825 |
| `loan_amount` | 15,247,252 | 14,946,063 |
| `residential_assets_value` | 7,399,812 | 7,592,498 |
| `luxury_assets_value` | 15,016,604 | 15,306,944 |

These features may still contribute through **feature engineering** (e.g. loan-to-income ratio, total assets).

---

### Finding 3 — Severe Multicollinearity Among Financial Features
| Feature Pair | Correlation |
|---|---|
| `income_annum` ↔ `loan_amount` | **0.93** |
| `income_annum` ↔ `luxury_assets_value` | **0.93** |
| `income_annum` ↔ `bank_asset_value` | **0.85** |
| `loan_amount` ↔ `luxury_assets_value` | **0.94** |

Keeping all features inflates dimensionality without adding new signal — must address before modelling.

---

### Finding 4 — Education and Self-Employed Are Not Predictive
Both features are 50/50 balanced and show **identical approval rates** regardless of category:

| Feature | Category | Approval Rate |
|---|---|---|
| `education` | Graduate | 62.5% |
| `education` | Not Graduate | 62.0% |
| `self_employed` | Yes | 62.2% |
| `self_employed` | No | 62.2% |

Consider dropping these features if model performance is unchanged without them.

---

### Finding 5 — Right-Skewed Asset Distributions
| Feature | Skewness | Action Needed |
|---|---|---|
| `residential_assets_value` | +0.978 | Log-transform |
| `commercial_assets_value` | +0.958 | Log-transform |
| `bank_asset_value` | +0.561 | Log-transform |
| `income_annum` | −0.013 | None (near-normal) |
| `cibil_score` | −0.009 | None (near-normal) |

---

### Finding 6 — Outliers Are Minimal
| Feature | Outlier Count | % |
|---|---|---|
| `residential_assets_value` | 52 | 1.2% |
| `commercial_assets_value` | 37 | 0.9% |
| `bank_asset_value` | 5 | 0.1% |

Capping via IQR fences is sufficient — removal is not necessary.

---

### Finding 7 — Reclassified Features Are Uniformly Distributed
- `no_of_dependents` (0–5): ~16–18% per value — no dominant category
- `loan_term` (2–20, even steps): 9–11.5% per term — no strong preference

## Pre-Modelling Steps

Based on the EDA findings above, the following steps are recommended before training any model.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_model = df.copy()

# ── Step 1: Drop primary key (already excluded in dataset_overview) ───────────
df_model.drop(columns=["loan_id"], errors="ignore", inplace=True)

# ── Step 2: Encode target ─────────────────────────────────────────────────────
df_model["loan_status"] = (df_model["loan_status"].str.strip() == "Approved").astype(int)
# Approved → 1, Rejected → 0

# ── Step 3: Encode binary categoricals ───────────────────────────────────────
df_model["education"]     = (df_model["education"].str.strip()     == "Graduate").astype(int)
df_model["self_employed"] = (df_model["self_employed"].str.strip() == "Yes").astype(int)

# ── Step 4: Feature engineering — address multicollinearity ──────────────────
df_model["total_assets"]   = (df_model["residential_assets_value"]
                             + df_model["commercial_assets_value"]
                             + df_model["luxury_assets_value"]
                             + df_model["bank_asset_value"])
df_model["loan_to_income"] = df_model["loan_amount"] / df_model["income_annum"]
df_model["asset_to_loan"]  = df_model["total_assets"] / df_model["loan_amount"]

# Drop highly correlated originals (0.93+ correlation with income_annum)
df_model.drop(columns=["loan_amount", "luxury_assets_value", "bank_asset_value"],
              inplace=True)

# ── Step 5: Cap outliers before log-transform ─────────────────────────────────
skewed_cols = ["residential_assets_value", "commercial_assets_value"]
for col in skewed_cols:
    q1, q3 = df_model[col].quantile(0.25), df_model[col].quantile(0.75)
    iqr    = q3 - q1
    df_model[col] = df_model[col].clip(lower=q1 - 1.5*iqr, upper=q3 + 1.5*iqr)

# ── Step 6: Log-transform right-skewed asset columns ─────────────────────────
for col in skewed_cols:
    df_model[col] = np.log1p(df_model[col])   # log1p handles zero values safely

# ── Step 7: Train / test split with stratification ───────────────────────────
X = df_model.drop(columns=["loan_status"])
y = df_model["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train target balance:\n{y_train.value_counts(normalize=True).round(3)}")

# ── Step 8: Feature scaling (required for linear models / SVM / KNN) ──────────
scale_cols = ["income_annum", "cibil_score", "residential_assets_value",
              "commercial_assets_value", "total_assets",
              "loan_to_income", "asset_to_loan"]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])    # transform only — no fit

print("\nFinal feature set:")
print(X_train.columns.tolist())